# CRM Sales Reports: Tables → Charts → PDF

Generate beautiful reports from CRM sales data using existing
`siege_utilities.reporting` primitives.

Sections:
1. Setup and data pull
2. Pipeline visualization
3. Activity time series
4. PDF report generation

In [ ]:
import os
import logging

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(name)s %(message)s")

## 1. Setup and Data Pull

In [ ]:
from siege_utilities.connectors import SalesforceConnector

sf = SalesforceConnector(
    client_id=os.environ.get("SF_CLIENT_ID", ""),
    client_secret=os.environ.get("SF_CLIENT_SECRET", ""),
    username=os.environ.get("SF_USERNAME"),
    password=os.environ.get("SF_PASSWORD"),
    security_token=os.environ.get("SF_SECURITY_TOKEN"),
)
sf.authenticate()

opportunities = sf.get_objects("Opportunity")
accounts = sf.get_objects("Account", limit=200)
print(f"Opportunities: {len(opportunities)}, Accounts: {len(accounts)}")

## 2. Pipeline Visualization

Use the pipeline adapter + chart generator for a sales funnel.

In [ ]:
from siege_utilities.connectors import pipeline_adapter
from siege_utilities.reporting import ChartGenerator

# Prepare pipeline data
pipeline = pipeline_adapter(
    opportunities,
    stage_column="StageName",
    value_column="Amount",
)
pipeline

In [ ]:
from siege_utilities.reporting import create_bar_chart

# Pipeline as horizontal bar chart
fig = create_bar_chart(
    data=pipeline,
    x_column="total_value",
    y_column="stage",
    title="Sales Pipeline by Stage",
    orientation="horizontal",
)
fig

## 3. Activity Time Series

Visualize opportunity creation over time.

In [ ]:
from siege_utilities.connectors import timeseries_adapter
from siege_utilities.reporting import create_line_chart

# Opportunities created over time
ts_data = timeseries_adapter(
    opportunities,
    timestamp_column="CreatedDate",
    freq="W",
    agg="count",
)

if not ts_data.empty:
    fig = create_line_chart(
        data=ts_data,
        x_column="date",
        y_column="value",
        title="Opportunities Created per Week",
    )
    fig
else:
    print("No timestamp data available for time series")

## 4. Top Accounts Table

In [ ]:
from siege_utilities.connectors import tabular_adapter

top_accounts = tabular_adapter(
    accounts,
    columns=["Name", "Industry", "AnnualRevenue", "NumberOfEmployees", "BillingCity", "BillingState"],
    rename={
        "AnnualRevenue": "Annual Revenue",
        "NumberOfEmployees": "Employees",
        "BillingCity": "City",
        "BillingState": "State",
    },
    sort_by="Annual Revenue",
    ascending=False,
    limit=25,
)
top_accounts

## 5. PDF Report Generation

In [ ]:
from siege_utilities.reporting import ReportGenerator

output_dir = os.environ.get("REPORT_OUTPUT_DIR", "output")
os.makedirs(output_dir, exist_ok=True)

report = ReportGenerator(
    title="Sales Pipeline Report",
    subtitle="Generated from Salesforce CRM Data",
    output_dir=output_dir,
)

# Add pipeline summary
report.add_section(
    title="Pipeline Summary",
    content="Sales pipeline by stage showing deal count and total value.",
    table_data=pipeline,
)

# Add top accounts
report.add_section(
    title="Top Accounts by Revenue",
    content="Top 25 accounts ranked by annual revenue.",
    table_data=top_accounts,
)

output_path = report.generate()
print(f"Report generated: {output_path}")

In [ ]:
sf.close()
print("Done.")